# 🌐 Khám Phá & Soát Lỗi Toàn Văn Dữ Liệu Gốc Theo Từng Domain
### ViBioMIR / Road to AI 2026 (R2AI) — Raw Medical Corpus by Domain Inspector

> **Mục đích của Notebook:**
> - Xem lại **100% văn bản nguyên gốc chưa tách chunk** (raw text before chunking) của toàn bộ kho dữ liệu (~381.325 bài viết).
> - Phân loại và lọc theo **từng domain nguồn** (Vinmec, Medlatec, Nhà thuốc Long Châu, Báo Sức khỏe & Đời sống, Bách khoa y học Trung Quốc,...).
> - Kiểm tra chất lượng cào dữ liệu (data scraping quality), rà soát lỗi font chữ (Mojibake) và phát hiện rác bản quyền / hotline.
> - Tích hợp CSS tự động ngắt dòng văn bản (`white-space: pre-wrap`) và màu chữ tương phản cao, xem mượt mà trên cả giao diện sáng lẫn tối.

In [ ]:
# 1. Khởi tạo môi trường & Cấu hình giao diện chống tràn dòng (Text-Wrapping)
import os
import sys
from pathlib import Path

# Tự động phát hiện và đặt Current Working Directory về thư mục gốc dự án
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"📁 Thư mục làm việc: {PROJECT_ROOT.resolve()}")

# Nhúng CSS toàn cục: Ép ngắt dòng tự động (word-wrap) và chuẩn hóa màu tương phản
from IPython.display import HTML, display

css_style = """
<style>
    .output_text pre, .output_subarea pre, div.output_area pre {
        white-space: pre-wrap !important;
        word-wrap: break-word !important;
        word-break: break-word !important;
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif !important;
    }
    .domain-badge {
        display: inline-block;
        padding: 3px 8px;
        border-radius: 4px;
        font-weight: 600;
        font-size: 12px;
        margin-right: 6px;
    }
</style>
"""
display(HTML(css_style))
print("✅ Đã kích hoạt CSS Text-Wrapping & Contrast Safeguard thành công!")

--- 
## 📊 1. Thống Kê Tổng Quan & Danh Mục Domain Nguồn (Domain Catalog)

Tải danh mục toàn bộ **89+ domains** đã cào về với số lượng bài viết, độ dài trung bình và ngôn ngữ tương ứng.

In [ ]:
import pandas as pd
from src.retrieval.doc_reader import get_domains_catalog

# Nạp danh mục domain (quét nhanh PyArrow Parquet Corpus trong < 0.5s)
catalog_df = get_domains_catalog()

total_articles = catalog_df["article_count"].sum()
total_domains = catalog_df["domain"].nunique()
vi_articles = catalog_df[catalog_df["lang"] == "vi"]["article_count"].sum()
zh_articles = catalog_df[catalog_df["lang"] == "zh"]["article_count"].sum()

display(HTML(f"""
<div style='display:flex; gap:16px; margin: 12px 0; flex-wrap:wrap;'>
    <div style='background:#f8fafc; border:1px solid #cbd5e1; border-radius:8px; padding:12px 18px; flex:1; min-width:160px;'>
        <div style='font-size:12px; color:#64748b; font-weight:bold;'>TỔNG SỐ BÀI VIẾT GỐC</div>
        <div style='font-size:24px; font-weight:bold; color:#0f172a;'>{total_articles:,}</div>
    </div>
    <div style='background:#eff6ff; border:1px solid #bfdbfe; border-radius:8px; padding:12px 18px; flex:1; min-width:160px;'>
        <div style='font-size:12px; color:#1d4ed8; font-weight:bold;'>SỐ LƯỢNG DOMAIN</div>
        <div style='font-size:24px; font-weight:bold; color:#1e40af;'>{total_domains} nguồn</div>
    </div>
    <div style='background:#f0fdf4; border:1px solid #bbf7d0; border-radius:8px; padding:12px 18px; flex:1; min-width:160px;'>
        <div style='font-size:12px; color:#15803d; font-weight:bold;'>BÀI TIẾNG VIỆT (VI)</div>
        <div style='font-size:24px; font-weight:bold; color:#166534;'>{vi_articles:,} bài</div>
    </div>
    <div style='background:#fefce8; border:1px solid #fef08a; border-radius:8px; padding:12px 18px; flex:1; min-width:160px;'>
        <div style='font-size:12px; color:#a16207; font-weight:bold;'>BÀI TIẾNG TRUNG (ZH)</div>
        <div style='font-size:24px; font-weight:bold; color:#854d0e;'>{zh_articles:,} bài</div>
    </div>
</div>
"""))

# Hiển thị Top 20 domain nhiều bài viết nhất
print("🏆 Top 20 Domain chiếm tỉ trọng lớn nhất trong kho dữ liệu:")
catalog_display = catalog_df.copy()
catalog_display["avg_char_length"] = catalog_display["avg_char_length"].round(0).astype(int)
catalog_display.columns = ["Tên Domain", "Ngôn ngữ", "Số bài viết", "Độ dài ký tự TB", "Tiêu đề mẫu"]
catalog_display.head(20).reset_index(drop=True)

--- 
## 🔬 2. Bộ Lọc & Trình Đọc Dữ Liệu Tương Tác Theo Domain (Interactive Inspector)

Chọn domain từ danh sách thả xuống, gõ từ khóa lọc (nếu muốn), hoặc bấm **🎲 Lấy ngẫu nhiên** để đọc toàn văn bài viết gốc trước khi bị chia nhỏ thành các chunks.

In [ ]:
import ipywidgets as widgets
from src.retrieval.doc_reader import (
    get_documents_by_domain,
    get_random_doc_by_domain,
    display_raw_document,
)

# Chuẩn bị danh sách các domain có trong catalog
all_domains = sorted(catalog_df["domain"].unique())

# Khởi tạo các Widgets tương tác
domain_dropdown = widgets.Dropdown(
    options=all_domains,
    value="www.vinmec.com" if "www.vinmec.com" in all_domains else all_domains[0],
    description="Domain:",
    layout=widgets.Layout(width="450px"),
)

kw_input = widgets.TextInput(
    value="",
    placeholder="Gõ từ khóa để lọc bài (ví dụ: dạ dày, sốt, gan, tiểu cầu...)",
    description="Từ khóa:",
    layout=widgets.Layout(width="450px"),
)

btn_load = widgets.Button(
    description="📖 Xem Bài Viết Đầu Tiên",
    button_style="info",
    icon="book",
    layout=widgets.Layout(width="200px"),
)

btn_random = widgets.Button(
    description="🎲 Lấy Ngẫu Nhiên Bài Mới",
    button_style="success",
    icon="random",
    layout=widgets.Layout(width="220px"),
)

out_view = widgets.Output()

def on_load_click(b):
    with out_view:
        out_view.clear_output()
        docs = get_documents_by_domain(domain_dropdown.value, limit=1, keyword_filter=kw_input.value)
        if docs:
            display_raw_document(docs[0])
        else:
            display(HTML(f"<b style='color:#dc2626;'>Không tìm thấy bài viết nào trong domain '{domain_dropdown.value}' khớp với từ khóa '{kw_input.value}'!</b>"))

def on_random_click(b):
    with out_view:
        out_view.clear_output()
        doc = get_random_doc_by_domain(domain_dropdown.value)
        if doc:
            display_raw_document(doc)
        else:
            display(HTML(f"<b style='color:#dc2626;'>Không lấy được tài liệu từ domain {domain_dropdown.value}!</b>"))

btn_load.on_click(on_load_click)
btn_random.on_click(on_random_click)

# Hiển thị khung điều khiển và kết quả
box_controls = widgets.VBox([
    widgets.HBox([domain_dropdown, kw_input]),
    widgets.HBox([btn_load, btn_random]),
])

display(box_controls)
display(out_view)

# Tự động nạp mẫu đầu tiên khi mở notebook
with out_view:
    sample_doc = get_documents_by_domain("www.vinmec.com", limit=1)
    if sample_doc:
        display_raw_document(sample_doc[0])

--- 
## 🔍 3. Tra Cứu Toàn Văn Dữ Liệu Gốc Bằng Doc ID Bất Kỳ

Nhập mã `doc_id` cụ thể để kiểm tra xem tài liệu đó thuộc domain nào, nội dung nguyên bản ra sao trước khi được index.

In [ ]:
import pyarrow.dataset as ds

def lookup_raw_doc_by_id(doc_id: int | str):
    """Tra cứu cực nhanh tài liệu gốc từ Parquet Corpus bằng doc_id."""
    try:
        target_id = int(str(doc_id).strip())
    except ValueError:
        display(HTML(f"<b style='color:#dc2626;'>Mã Doc ID không hợp lệ: {doc_id}</b>"))
        return
        
    p_dir = Path("data/processed/parquet_corpus")
    dataset = ds.dataset(p_dir, format="parquet")
    scanner = dataset.scanner(
        columns=["doc_id", "title", "text", "url", "lang", "domain", "char_count"],
        filter=(ds.field("doc_id") == target_id),
    )
    rows = scanner.to_table().to_pylist()
    if rows:
        display_raw_document(rows[0])
    else:
        display(HTML(f"<b style='color:#dc2626;'>Không tìm thấy Doc ID {target_id} trong 381.325 bài viết Parquet!</b>"))

# Thử tra cứu Doc ID mẫu: 733125 (Bài viết Vinmec), 2989609 (Bài viết tiếng Trung)
lookup_raw_doc_by_id(733125)

--- 
## 🧹 4. Soát Lỗi Rác & Đánh Giá Mức Độ Sạch Của Từng Domain (Boilerplate Audit)

So sánh bài viết nguyên gốc với bài viết sau khi chạy qua bộ chuẩn hóa Unicode và 6 bộ Regex làm sạch rác (`src.ingestion.cleaner`).

In [ ]:
from src.ingestion.cleaner import normalize_text, clean_boilerplate

# Chọn một bài viết bất kỳ từ báo Sức khỏe & Đời sống hoặc Báo Thanh Niên để kiểm tra rác footer
test_docs = get_documents_by_domain("suckhoedoisong.vn", limit=1)

if test_docs:
    d = test_docs[0]
    raw_t = d["text"]
    cleaned_t = normalize_text(raw_t, strip_boilerplate=True)
    
    chars_removed = len(raw_t) - len(cleaned_t)
    percent_removed = (chars_removed / len(raw_t) * 100) if len(raw_t) > 0 else 0
    
    display(HTML(f"""
    <div style='border:1px solid #cbd5e1; border-radius:8px; padding:16px; margin:12px 0; background:#ffffff;'>
        <h4 style='color:#1d4ed8; margin-top:0;'>🧹 Kết Quả Đánh Giá Làm Sạch Rác Cho Domain: <code>{d.get('domain')}</code></h4>
        <div style='font-size:13px; color:#475569; margin-bottom:12px;'>
            Doc ID: <b>{d.get('doc_id')}</b> | Độ dài gốc: <b>{len(raw_t):,}</b> ký tự | Sau làm sạch: <b>{len(cleaned_t):,}</b> ký tự
            | Rác đã lược bỏ: <b style='color:#dc2626;'>{chars_removed:,} ký tự ({percent_removed:.1f}%)</b>
        </div>
        
        <div style='display:grid; grid-template-columns: 1fr 1fr; gap:16px;'>
            <div>
                <div style='font-weight:bold; color:#b91c1c; font-size:13px; margin-bottom:6px;'>📜 100 Ký Tự Cuối Của Văn Bản Gốc (Thường chứa Hotline / Copyright):</div>
                <div style='background:#fef2f2; border:1px solid #fecaca; color:#0f172a; padding:10px; border-radius:6px; font-size:13px; white-space:pre-wrap;'>
                    {raw_t[-250:]}
                </div>
            </div>
            <div>
                <div style='font-weight:bold; color:#15803d; font-size:13px; margin-bottom:6px;'>✨ 100 Ký Tự Cuối Sau Khi Qua DocumentCleaner:</div>
                <div style='background:#f0fdf4; border:1px solid #bbf7d0; color:#0f172a; padding:10px; border-radius:6px; font-size:13px; white-space:pre-wrap;'>
                    {cleaned_t[-250:]}
                </div>
            </div>
        </div>
    </div>
    """))

--- 
## 💾 5. Trích Xuất & Xuất Mẫu Dữ Liệu Theo Domain Ra File

Bạn có thể lấy nhanh $N$ bài viết bất kỳ của một domain để xuất ra file CSV hoặc JSONL lưu trữ ngoại tuyến.

In [ ]:
def export_domain_sample(domain_name: str, sample_size: int = 50, output_csv: str = None):
    """Lấy danh sách các bài viết gốc của domain và xuất ra DataFrame / CSV."""
    docs = get_documents_by_domain(domain_name, limit=sample_size)
    if not docs:
        print(f"❌ Không tìm thấy dữ liệu cho domain: {domain_name}")
        return None
        
    export_df = pd.DataFrame(docs)
    export_df["preview"] = export_df["text"].apply(lambda t: t[:150] + "...")
    
    if output_csv:
        out_path = Path(output_csv)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        export_df.to_csv(out_path, index=False, encoding="utf-8-sig")
        print(f"✅ Đã xuất {len(export_df)} bài viết của {domain_name} vào: {out_path.resolve()}")
        
    return export_df[["doc_id", "title", "lang", "char_count", "preview"]]

# Xuất mẫu 10 bài của Vinmec
sample_view = export_domain_sample("www.vinmec.com", sample_size=10)
sample_view